In [0]:
# ดึงข้อมูลจากตาราง workspace.default.superstore
bronze_df = spark.read.table("workspace.default.superstore")

# แสดงผลข้อมูล 5 บรรทัดแรก
print(f"จำนวนข้อมูลทั้งหมดใน Bronze Layer: {bronze_df.count()} แถว")
display(bronze_df.limit(5))

In [0]:
# เปลี่ยนชื่อคอลัมน์โดยแทนที่ "ช่องว่าง" ด้วย "_" (เช่น Order ID -> Order_ID)
silver_df = bronze_df
for col_name in silver_df.columns:
    silver_df = silver_df.withColumnRenamed(col_name, col_name.replace(" ", "_"))

# กรองเฉพาะข้อมูลที่รหัสลูกค้า (Customer_ID) ไม่เป็นค่าว่าง (Null)
silver_df = silver_df.filter(silver_df.Customer_ID.isNotNull())

# แปลง DataFrame ให้เป็น Temporary View เพื่อให้เราสามารถใช้คำสั่ง SQL วิเคราะห์ต่อได้
silver_df.createOrReplaceTempView("silver_superstore")

print("ทำความสะอาดข้อมูลเสร็จสิ้น")
display(silver_df.limit(5))

In [0]:
%sql
-- วิเคราะห์ยอดพรีออเดอร์และจำนวนลูกค้า แยกตามภูมิภาค (Region) และหมวดหมู่สินค้า (Category)
SELECT 
    Region,
    Category,
    COUNT(DISTINCT Order_ID) AS Total_Orders,
    COUNT(DISTINCT Customer_ID) AS Total_Unique_Customers
FROM silver_superstore
GROUP BY Region, Category
ORDER BY Region ASC, Total_Orders DESC;